# Machine Learning Pipeline with Daglab\n\nThis notebook demonstrates how to build ML pipelines with Dagster and daglab integration.

In [ ]:
# Import required libraries\nfrom daglab import DagsterClient, asset_from_notebook\nimport pandas as pd\nimport numpy as np\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.ensemble import RandomForestClassifier\nfrom sklearn.metrics import accuracy_score, classification_report\nimport matplotlib.pyplot as plt\nimport seaborn as sns

## Load and Explore Data

In [ ]:
# Generate synthetic dataset for demonstration\nnp.random.seed(42)\nn_samples = 1000\nn_features = 10\n\nX = np.random.randn(n_samples, n_features)\n# Create a non-linear relationship\ny = (X[:, 0] * X[:, 1] + X[:, 2]**2 - X[:, 3] + np.random.randn(n_samples) * 0.1 > 0).astype(int)\n\n# Create DataFrame\nfeature_names = [f'feature_{i}' for i in range(n_features)]\ndf = pd.DataFrame(X, columns=feature_names)\ndf['target'] = y\n\nprint(f"Dataset shape: {df.shape}")\nprint(f"Class distribution:\\n{df['target'].value_counts()}")\ndf.head()

## Data Exploration and Visualization

In [ ]:
# Visualize feature distributions\nfig, axes = plt.subplots(2, 3, figsize=(15, 10))\naxes = axes.flatten()\n\nfor i, col in enumerate(feature_names[:6]):\n    df[col].hist(bins=30, ax=axes[i])\n    axes[i].set_title(f'Distribution of {col}')\n    axes[i].set_xlabel(col)\n    axes[i].set_ylabel('Frequency')\n\nplt.tight_layout()\nplt.show()

In [ ]:
# Correlation matrix\nplt.figure(figsize=(12, 10))\ncorrelation_matrix = df.corr()\nsns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f')\nplt.title('Feature Correlation Matrix')\nplt.show()

## Feature Engineering

In [ ]:
# Create engineered features\ndf_features = df.copy()\n\n# Polynomial features\ndf_features['feature_0_squared'] = df_features['feature_0'] ** 2\ndf_features['feature_1_squared'] = df_features['feature_1'] ** 2\n\n# Interaction features\ndf_features['interaction_0_1'] = df_features['feature_0'] * df_features['feature_1']\ndf_features['interaction_2_3'] = df_features['feature_2'] * df_features['feature_3']\n\n# Statistical features\ndf_features['mean_features'] = df_features[feature_names].mean(axis=1)\ndf_features['std_features'] = df_features[feature_names].std(axis=1)\n\nprint(f"Features after engineering: {df_features.shape[1] - 1}")  # -1 for target\ndf_features.head()

## Model Training

In [ ]:
# Prepare data for training\nX = df_features.drop('target', axis=1)\ny = df_features['target']\n\n# Split data\nX_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)\n\n# Train Random Forest model\nrf_model = RandomForestClassifier(\n    n_estimators=100,\n    max_depth=10,\n    random_state=42,\n    n_jobs=-1\n)\n\nrf_model.fit(X_train, y_train)\n\n# Predictions\ny_train_pred = rf_model.predict(X_train)\ny_test_pred = rf_model.predict(X_test)\n\n# Accuracy\ntrain_accuracy = accuracy_score(y_train, y_train_pred)\ntest_accuracy = accuracy_score(y_test, y_test_pred)\n\nprint(f"Train Accuracy: {train_accuracy:.4f}")\nprint(f"Test Accuracy: {test_accuracy:.4f}")\n\n# Feature importance\nfeature_importance = pd.DataFrame({\n    'feature': X.columns,\n    'importance': rf_model.feature_importances_\n}).sort_values('importance', ascending=False)\n\n# Visualize top features\nplt.figure(figsize=(10, 8))\ntop_features = feature_importance.head(10)\nplt.barh(top_features['feature'], top_features['importance'])\nplt.xlabel('Importance')\nplt.title('Top 10 Feature Importances')\nplt.gca().invert_yaxis()\nplt.show()\n\n# Output model for Dagster asset\nmodel_output = {\n    'model': rf_model,\n    'train_accuracy': train_accuracy,\n    'test_accuracy': test_accuracy,\n    'feature_importance': feature_importance\n}

## Model Evaluation

In [ ]:
# Detailed classification report\nprint("Classification Report:\\n")\nprint(classification_report(y_test, y_test_pred))\n\n# Confusion matrix\nfrom sklearn.metrics import confusion_matrix\nimport seaborn as sns\n\ncm = confusion_matrix(y_test, y_test_pred)\nplt.figure(figsize=(8, 6))\nsns.heatmap(cm, annot=True, fmt='d', cmap='Blues')\nplt.xlabel('Predicted')\nplt.ylabel('Actual')\nplt.title('Confusion Matrix')\nplt.show()\n\n# Create evaluation summary\nevaluation_summary = {\n    'test_accuracy': test_accuracy,\n    'classification_report': classification_report(y_test, y_test_pred, output_dict=True),\n    'confusion_matrix': cm.tolist(),\n    'top_features': feature_importance.head(5).to_dict('records')\n}\n\nevaluation_summary

## Export Model for Production\n\nThis cell demonstrates how to save the model for use in production pipelines.

In [ ]:
import joblib\nfrom datetime import datetime\n\n# Save model\nmodel_path = f"models/rf_model_{datetime.now().strftime('%Y%m%d_%H%M%S')}.joblib"\n# joblib.dump(rf_model, model_path)\nprint(f"Model would be saved to: {model_path}")\n\n# Save model metadata\nmetadata = {\n    'model_type': 'RandomForestClassifier',\n    'features': list(X.columns),\n    'train_accuracy': train_accuracy,\n    'test_accuracy': test_accuracy,\n    'training_date': datetime.now().isoformat(),\n    'n_samples_train': len(X_train),\n    'n_samples_test': len(X_test)\n}\n\nprint("\\nModel Metadata:")\nfor key, value in metadata.items():\n    print(f"  {key}: {value}")

## Next Steps\n\n1. Use `daglab sync` to convert these notebook cells into Dagster assets\n2. The tagged cells will become:\n   - `raw_dataset` - Data generation asset\n   - `engineered_features` - Feature engineering asset\n   - `trained_model` - Model training asset\n   - `model_evaluation` - Evaluation metrics asset\n3. Run the ML pipeline in Dagster UI\n4. Monitor asset materialization and lineage